In [1]:
%pip install d2l --no-deps --quiet

In [2]:
# 导入 Python 内置的数学模块，提供 sqrt（开平方）、pi 等数学函数
import math

# 导入 PyTorch 深度学习框架主库，提供张量运算、自动求导等核心功能
import torch

# 导入 PyTorch 的神经网络模块，包含 Linear、Dropout、Module 等常用网络组件
from torch import nn

# 从 d2l 库中导入针对 PyTorch 的工具集，包含 DotProductAttention（点积注意力）等辅助类
from d2l import torch as d2l

In [3]:
#@save
def transpose_qkv(X, num_heads):
    """为了多注意力头的并行计算而变换形状

    将输入张量 X 从 (batch_size, 序列长度, num_hiddens)
    变换为 (batch_size * num_heads, 序列长度, num_hiddens / num_heads)，
    使得多个注意力头可以在同一个矩阵运算中并行计算。
    """
    # X 的初始形状: (batch_size, 查询/键/值的个数, num_hiddens)
    # 将最后一维 num_hiddens 拆分为 (num_heads, num_hiddens/num_heads)
    # -1 让 PyTorch 自动推断每个头的隐藏维度大小 = num_hiddens // num_heads
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)
    # 此时 X 形状: (batch_size, 序列长度, num_heads, num_hiddens/num_heads)

    # 用 permute 调整维度顺序，把 num_heads 移到第2维（紧跟 batch_size）
    # 目的是让后续把 batch_size 和 num_heads 合并为一个维度，实现并行计算
    X = X.permute(0, 2, 1, 3)
    # 此时 X 形状: (batch_size, num_heads, 序列长度, num_hiddens/num_heads)

    # 将前两个维度合并：batch_size * num_heads 作为新的"批量"维度
    # 这样注意力计算时，每个头都被视为一个独立的样本，可以批量并行运算
    return X.reshape(-1, X.shape[2], X.shape[3])
    # 返回形状: (batch_size * num_heads, 序列长度, num_hiddens/num_heads)


#@save
def transpose_output(X, num_heads):
    """逆转 transpose_qkv 函数的操作

    将多头注意力输出从 (batch_size * num_heads, 序列长度, num_hiddens/num_heads)
    还原为 (batch_size, 序列长度, num_hiddens)，完成多头拼接。
    """
    # X 输入形状: (batch_size * num_heads, 序列长度, num_hiddens/num_heads)
    # 先把第0维拆回 (batch_size, num_heads)，恢复四维张量
    X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
    # 此时 X 形状: (batch_size, num_heads, 序列长度, num_hiddens/num_heads)

    # 调换维度顺序，把 num_heads 移回第3维，序列长度提到第2维
    X = X.permute(0, 2, 1, 3)
    # 此时 X 形状: (batch_size, 序列长度, num_heads, num_hiddens/num_heads)

    # 将最后两维合并：num_heads * (num_hiddens/num_heads) = num_hiddens
    # 相当于把所有头的输出拼接在一起，恢复原始隐藏维度
    return X.reshape(X.shape[0], X.shape[1], -1)
    # 返回形状: (batch_size, 序列长度, num_hiddens)

In [4]:
#@save
class MultiHeadAttention(nn.Module):
    """多头注意力模块

    核心思想：将查询(Q)、键(K)、值(V)分别投影到 num_heads 个子空间，
    在每个子空间独立计算注意力（缩放点积），最后将所有头的结果拼接并线性变换输出。
    这样模型可以同时关注来自不同表示子空间的信息。
    """

    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 num_heads, dropout, bias=False, **kwargs):
        """初始化多头注意力模块

        参数说明：
            key_size   : 键向量的特征维度
            query_size : 查询向量的特征维度
            value_size : 值向量的特征维度
            num_hiddens: 内部隐藏层的总维度（也是输出维度），每个头的维度 = num_hiddens / num_heads
            num_heads  : 注意力头的数量
            dropout    : Dropout 概率，用于注意力权重的正则化，防止过拟合
            bias       : 线性层是否使用偏置项，默认 False（Transformer 原论文不使用偏置）
            **kwargs   : 传递给父类 nn.Module 的额外参数
        """
        # 调用父类 nn.Module 的初始化方法，完成 PyTorch 模块的基础注册
        super(MultiHeadAttention, self).__init__(**kwargs)

        # 保存头的数量，后续 forward 和形状变换都需要用到
        self.num_heads = num_heads

        # 使用 d2l 提供的缩放点积注意力（DotProductAttention）作为每个头的注意力计算单元
        # DotProductAttention 内部会对注意力分数除以 sqrt(d_k) 并应用 softmax + dropout
        self.attention = d2l.DotProductAttention(dropout)

        # 查询的线性投影层：将输入查询从 query_size 维映射到 num_hiddens 维
        # 每个头实际使用其中 num_hiddens/num_heads 维（通过 reshape 拆分，不是 num_heads 个独立线性层）
        self.W_q = nn.Linear(query_size, num_hiddens, bias=bias)

        # 键的线性投影层：将输入键从 key_size 维映射到 num_hiddens 维
        self.W_k = nn.Linear(key_size, num_hiddens, bias=bias)

        # 值的线性投影层：将输入值从 value_size 维映射到 num_hiddens 维
        self.W_v = nn.Linear(value_size, num_hiddens, bias=bias)

        # 输出的线性投影层：将所有头拼接后的 num_hiddens 维再投影回 num_hiddens 维
        # 这一步融合了各头捕获的不同子空间信息
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)

    def forward(self, queries, keys, values, valid_lens):
        """前向传播

        参数说明：
            queries   : 查询张量，形状 (batch_size, 查询个数, query_size)
            keys      : 键张量，  形状 (batch_size, 键值对个数, key_size)
            values    : 值张量，  形状 (batch_size, 键值对个数, value_size)
            valid_lens: 有效长度，用于掩码填充位置，避免注意力关注到 padding 部分
                        形状为 (batch_size,) 或 (batch_size, 查询个数)
        """
        # ---- 第一步：线性投影 + 形状变换，为并行多头计算做准备 ----

        # 对查询做线性投影：(batch_size, 查询个数, num_hiddens)
        # 再通过 transpose_qkv 变为: (batch_size * num_heads, 查询个数, num_hiddens/num_heads)
        queries = transpose_qkv(self.W_q(queries), self.num_heads)

        # 对键做线性投影，形状变换同上
        keys = transpose_qkv(self.W_k(keys), self.num_heads)

        # 对值做线性投影，形状变换同上
        values = transpose_qkv(self.W_v(values), self.num_heads)

        # ---- 第二步：处理有效长度，使每个头都能正确掩码 ----
        if valid_lens is not None:
            # valid_lens 原形状: (batch_size,) 或 (batch_size, 查询个数)
            # 由于形状变换后有效批量变为 batch_size * num_heads，
            # 需要把每个样本的 valid_lens 重复 num_heads 次
            # repeat_interleave 按元素重复：[a, b] -> [a, a, ..., b, b, ...]（每个重复 num_heads 次）
            # dim=0 沿第一个维度（批量维）重复
            valid_lens = torch.repeat_interleave(
                valid_lens, repeats=self.num_heads, dim=0)

        # ---- 第三步：计算缩放点积注意力 ----
        # queries/keys/values 的形状均为: (batch_size*num_heads, 序列长度, num_hiddens/num_heads)
        # DotProductAttention 将计算注意力分数、softmax、dropout，并加权求和 values
        # output 形状: (batch_size * num_heads, 查询个数, num_hiddens/num_heads)
        output = self.attention(queries, keys, values, valid_lens)

        # ---- 第四步：逆变换，将多头输出拼接 ----
        # transpose_output 将 (batch_size*num_heads, 查询个数, num_hiddens/num_heads)
        # 还原为 (batch_size, 查询个数, num_hiddens)，相当于拼接所有头的结果
        output_concat = transpose_output(output, self.num_heads)

        # ---- 第五步：最终线性投影，融合多头信息 ----
        # W_o 将拼接后的 num_hiddens 维输出再做一次线性变换，映射回 num_hiddens 维
        # 返回形状: (batch_size, 查询个数, num_hiddens)
        return self.W_o(output_concat)

In [5]:
# ---- 超参数设置 ----
# num_hiddens: 模型内部的总隐藏维度为 100
# num_heads  : 使用 5 个注意力头，因此每个头的维度 = 100 / 5 = 20
num_hiddens, num_heads = 100, 5

# 创建多头注意力实例：
# 参数依次为: key_size=100, query_size=100, value_size=100（键/查询/值输入维度均为100）
#              num_hiddens=100（内部隐藏/输出维度）
#              num_heads=5（5个头）
#              dropout=0.5（训练时对注意力权重以50%概率随机置零，防止过拟合）
attention = MultiHeadAttention(num_hiddens, num_hiddens, num_hiddens,
                               num_hiddens, num_heads, 0.5)

# 调用 .eval() 将模型切换到评估模式：
# 在评估模式下 Dropout 层不会随机丢弃神经元（dropout 失效），BatchNorm 使用运行统计量
# 只有需要评估或推理时才调用 eval()，训练时需调用 model.train()
attention.eval()

MultiHeadAttention(
  (attention): DotProductAttention(
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (W_q): Linear(in_features=100, out_features=100, bias=False)
  (W_k): Linear(in_features=100, out_features=100, bias=False)
  (W_v): Linear(in_features=100, out_features=100, bias=False)
  (W_o): Linear(in_features=100, out_features=100, bias=False)
)

In [6]:
# ---- 构造测试数据 ----
# batch_size: 一次处理 2 个序列样本
# num_queries: 每个样本有 4 个查询向量（如解码器的 4 个位置）
batch_size, num_queries = 2, 4

# num_kvpairs: 每个样本有 6 个键值对（如编码器输出的 6 个位置）
# valid_lens : 每个样本实际有效的键值对数量
#              第1个样本有效长度为3（只关注前3个键值对，第4~6个是 padding）
#              第2个样本有效长度为2（只关注前2个键值对）
num_kvpairs, valid_lens = 6, torch.tensor([3, 2])

# 构造查询张量 X：形状 (batch_size=2, num_queries=4, num_hiddens=100)，全1填充（仅用于测试形状）
X = torch.ones((batch_size, num_queries, num_hiddens))

# 构造键/值张量 Y：形状 (batch_size=2, num_kvpairs=6, num_hiddens=100)，全1填充（仅用于测试形状）
Y = torch.ones((batch_size, num_kvpairs, num_hiddens))

# 执行前向传播：
# queries=X（形状 2×4×100），keys=Y，values=Y（形状 2×6×100）
# 第二个 Y 既作为键也作为值（自注意力或交叉注意力均可，此处两者相同）
# valid_lens 指定每个样本的有效键值对数，模型会对超出长度的位置进行掩码（填充 -inf）
# 期望输出形状: (batch_size=2, num_queries=4, num_hiddens=100)
attention(X, Y, Y, valid_lens).shape

torch.Size([2, 4, 100])